# Rosters to Delta table

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

ACCOUNT    = "footballmanagerli"
SRC        = f"abfss://alsource@{ACCOUNT}.dfs.core.windows.net/rosters/"
DEST       = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/rosters/rosters"
CHECKPOINT = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/_checkpoints/rosters"
TABLE      = "nfl.rosters"

In [ ]:
roster_schema = StructType([
    StructField("teamId",      StringType()),
    StructField("position",    StringType()),
    StructField("playerId",    StringType()),
    StructField("playerName",  StringType()),
    StructField("jersey",      StringType()),
    StructField("age",         IntegerType()),
    StructField("height",      StringType()),
    StructField("weight",      StringType()),
    StructField("experience",  StringType()),
])

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS nfl")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
    teamId        STRING    COMMENT 'Team ID',
    position      STRING    COMMENT 'Player position',
    playerId      STRING    COMMENT 'Player ID (composite key with teamId)',
    playerName    STRING,
    jersey        STRING,
    age           INT,
    height        STRING,
    weight        STRING,
    experience    STRING,
    snapshot_at   TIMESTAMP COMMENT 'When the snapshot file was written',
    source_file   STRING,
    ingested_at   TIMESTAMP
)
USING DELTA
LOCATION '{DEST}'
""")

In [ ]:
raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .schema(roster_schema)
    .load(SRC)
    .select(
        "*",
        F.col("_metadata.file_path").alias("source_file"),
        F.col("_metadata.file_modification_time").alias("snapshot_at"),
    ))

rosters = (raw
    .select(
        F.col("teamId"),
        F.col("position"),
        F.col("playerId"),
        F.col("playerName"),
        F.col("jersey"),
        F.col("age"),
        F.col("height"),
        F.col("weight"),
        F.col("experience"),
        "snapshot_at",
        "source_file",
        F.current_timestamp().alias("ingested_at"),
    ))

In [ ]:
def upsert_batch(batch_df, batch_id):
    # Dedupe within the batch: keep the newest snapshot per player/team
    w = Window.partitionBy("teamId", "playerId").orderBy(F.col("snapshot_at").desc())
    latest = (batch_df
        .withColumn("_rn", F.row_number().over(w))
        .filter("_rn = 1")
        .drop("_rn"))

    # Merge into the table: insert new records, update if snapshot is newer
    (DeltaTable.forName(spark, TABLE).alias("t")
        .merge(latest.alias("s"), "t.teamId = s.teamId AND t.playerId = s.playerId")
        .whenMatchedUpdateAll(condition="s.snapshot_at > t.snapshot_at")
        .whenNotMatchedInsertAll()
        .execute())

(rosters.writeStream
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination())

In [ ]:
%sql
SELECT COUNT(*) AS rows FROM nfl.rosters;

SELECT teamId, position, playerName, jersey FROM nfl.rosters ORDER BY teamId, position, playerName LIMIT 20;